# Reading a `LibraryReport`: real deposits, no download

> ℹ️ **Illustrative tour of real `detect` outputs — _not_ benchmark evidence.**
> These are cached `LibraryReport` JSONs that `selexprep detect` produced on real
> public deposits, bundled here so you can see realistic behavior with **no
> network or HPC**. They show the report *format and the range of outcomes*, not
> a performance claim — primer-recovery *performance* on paper-documented
> deposits is established in **the Tier 1 scorecard** (`benchmarks/`).

A `LibraryReport` answers three questions: did `selexprep` find the library's
primers, how confident is it, and what should you do next? The three real
deposits below span the outcome range.

## The fields that matter

- **`status`** — `HIGH` / `MEDIUM` / `LOW` / `UNABLE_TO_INFER`: overall confidence.
- **`extraction_mode`** — *biology*: `BOTH_PRIMERS_SINGLE_READ`, `FIVE_PRIME_ONLY`,
  `THREE_PRIME_ONLY`, `PAIRED_END_SPLIT_PRIMERS`, `UNABLE_TO_EXTRACT`.
- **`required_action`** — *workflow*: `NONE`, `MANUAL_PRIMERS_REQUIRED`, `READ_MERGING_RECOMMENDED`.
- **`full_insert_recovered`** — were both flanks found in a single read?
- **`known_adapter_hits`** — sequencing adapters seen (recorded, never silently called as primers).
- **`match_rate_*` / `position_consistency_*`** — how cleanly each constant sits at the read end.

Design point: `extraction_mode` (biology), `required_action` (workflow), and
`read_source` (file layout) are **separate axes** — read them together.

In [1]:
import json, pathlib

def show(acc):
    r = json.load(open(pathlib.Path("data") / f"{acc}.library_report.json"))
    keys = ["status", "extraction_mode", "read_source", "required_action",
            "full_insert_recovered", "primer_5p", "primer_3p",
            "match_rate_5p", "match_rate_3p", "n_length_mode", "known_adapter_hits"]
    print(f"# {acc}")
    for k in keys:
        print(f"  {k:22} {r[k]}")
    if r.get("failure_reason"):
        print(f"  {'failure_reason':22} {r['failure_reason']}")
    return r

## 1. A clean recovery — `HIGH`  (PRJNA615076, DNA whole-cell SELEX)

In [2]:
_ = show("PRJNA615076")

# PRJNA615076
  status                 HIGH
  extraction_mode        BOTH_PRIMERS_SINGLE_READ
  read_source            R1_AND_R2
  required_action        NONE
  full_insert_recovered  True
  primer_5p              TAGGGAAGAGAAGGACATATGAT
  primer_3p              TTGACTAGTACATGACCACTTGA
  match_rate_5p          0.9304520708300043
  match_rate_3p          0.9293465133131422
  n_length_mode          40
  known_adapter_hits     {'NEXTERA': 1854, 'TRUSEQ_R1': 0}


Both primers recovered, `extraction_mode = BOTH_PRIMERS_SINGLE_READ`,
`full_insert_recovered = True`, `required_action = NONE` → `extract` proceeds
with no manual input. (These happen to match the paper's primers — but checking
that is the Tier 1 scorecard's job, not this notebook's.)

## 2. A harder, partial case — `MEDIUM`  (PRJEB62495)

In [3]:
_ = show("PRJEB62495")

# PRJEB62495
  status                 MEDIUM
  extraction_mode        FIVE_PRIME_ONLY
  read_source            R1
  required_action        NONE
  full_insert_recovered  False
  primer_5p              GGCTTCTGGACTACCTATGC
  primer_3p              CGTGGTTACAGTCAGAGGACAGATT
  match_rate_5p          0.9730360686354617
  match_rate_3p          0.5349081521576846
  n_length_mode          40
  known_adapter_hits     {'NEXTERA': 0, 'TRUSEQ_R1': 410}


`extraction_mode = FIVE_PRIME_ONLY`: the 5' constant is clean
(`match_rate_5p ≈ 0.97`) but the 3' is noisy (`match_rate_3p ≈ 0.53`) — the
deposited reads carry a heterogeneous/extended 3' boundary, so `detect` reports
`MEDIUM` and a 5'-anchored extraction instead of inventing a clean 3'. This is
faithful-to-the-reads behavior, not an error.

## 3. A safe failure — `UNABLE_TO_INFER`  (PRJEB70964, adapter collision)

In [4]:
_ = show("PRJEB70964")

# PRJEB70964
  status                 UNABLE_TO_INFER
  extraction_mode        UNABLE_TO_EXTRACT
  read_source            R1
  required_action        MANUAL_PRIMERS_REQUIRED
  full_insert_recovered  False
  primer_5p              None
  primer_3p              None
  match_rate_5p          0.0
  match_rate_3p          0.0
  n_length_mode          35
  known_adapter_hits     {'NEXTERA': 8, 'TRUSEQ_R1': 66}
  failure_reason         Both primer match rates below 0.40 (5'=0.00, 3'=0.00)


This library's 5' constant is the reverse-complement of the TruSeq R1
adapter. `selexprep` blacklists known adapters as primer candidates, so instead
of "recovering" the adapter as a primer it returns `primer_5p/3p = null`,
`status = UNABLE_TO_INFER`, `required_action = MANUAL_PRIMERS_REQUIRED`, and
records what it saw in `known_adapter_hits`. **Downstream `extract` then
refuses** unless you pass `--override-primer-5p/-3p` — no silent miscalls.

## Takeaway
`HIGH → extract proceeds` · `MEDIUM → proceeds, mind the caveats` ·
`UNABLE_TO_INFER → refuses, asks for manual primers`. The safe-failure path —
refusing rather than fabricating a primer — is the property known-primer
pipelines can't offer. For recovery *performance* on paper-documented deposits,
see the Tier 1 scorecard.